<a href="https://colab.research.google.com/github/shadesvinay01/AI-saas/blob/main/AI_saas_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import io
import base64

print(f"✅ Gradio version: {gr.__version__}")
print(f"✅ Pandas version: {pd.__version__}")

# ============================================
# SIMPLE SAAS CLASS
# ============================================

class SimpleSaaS:
    def __init__(self):
        self.df = None

    def load_file(self, file):
        """Load CSV file"""
        if file is None:
            return "⚠️ No file uploaded", pd.DataFrame(), 0, 0, []

        try:
            self.df = pd.read_csv(file.name)
            preview = self.df.head(10)
            cols = list(self.df.columns)

            return (f"✅ Loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns",
                   preview, self.df.shape[0], self.df.shape[1], cols)
        except Exception as e:
            return f"❌ Error: {str(e)}", pd.DataFrame(), 0, 0, []

    def analyze_data(self):
        """Basic analysis"""
        if self.df is None:
            return "Please upload data first", None

        try:
            # Basic stats
            stats = self.df.describe().round(2)

            # Create plot
            plt.figure(figsize=(12, 8))

            numeric_cols = self.df.select_dtypes(include=[np.number]).columns
            if len(numeric_cols) > 0:
                n_cols = min(4, len(numeric_cols))
                for i, col in enumerate(numeric_cols[:n_cols]):
                    plt.subplot(2, 2, i+1)
                    self.df[col].hist(bins=20, color='skyblue', edgecolor='black')
                    plt.title(f'Distribution of {col}')
                    plt.xlabel(col)
                    plt.ylabel('Frequency')
                plt.tight_layout()

                # Save plot
                img_bytes = io.BytesIO()
                plt.savefig(img_bytes, format='png', dpi=100)
                img_bytes.seek(0)
                plt.close()

                # Convert to base64
                img_base64 = base64.b64encode(img_bytes.read()).decode()
                html_img = f'<img src="data:image/png;base64,{img_base64}" style="width:100%">'
            else:
                html_img = "<p>No numeric columns to plot</p>"

            result = f"""
### 📊 Analysis Results
- **Total Rows:** {self.df.shape[0]:,}
- **Total Columns:** {self.df.shape[1]}
- **Missing Values:** {self.df.isnull().sum().sum():,}
- **Duplicate Rows:** {self.df.duplicated().sum():,}
- **Numeric Columns:** {len(numeric_cols)}
- **Categorical Columns:** {len(self.df.select_dtypes(include=['object']).columns)}
            """

            return result, html_img
        except Exception as e:
            return f"Error: {str(e)}", None

    def run_ml(self, target_col, task_type):
        """Simple ML"""
        if self.df is None:
            return "Please upload data first", None, pd.DataFrame()

        try:
            if target_col not in self.df.columns:
                return f"Column '{target_col}' not found", None, pd.DataFrame()

            # Prepare data
            df_ml = self.df.copy()
            df_ml = df_ml.dropna(subset=[target_col])

            if len(df_ml) < 10:
                return "Need at least 10 samples", None, pd.DataFrame()

            # Encode categorical
            le = LabelEncoder()
            for col in df_ml.select_dtypes(include=['object']).columns:
                if col != target_col:
                    df_ml[col] = le.fit_transform(df_ml[col].astype(str))

            # Prepare target
            if df_ml[target_col].dtype == 'object':
                y = le.fit_transform(df_ml[target_col])
                is_classification = True
            else:
                y = df_ml[target_col].values
                is_classification = False

            # Features
            feature_cols = [c for c in df_ml.select_dtypes(include=[np.number]).columns
                           if c != target_col]

            if len(feature_cols) == 0:
                return "No features available", None, pd.DataFrame()

            X = df_ml[feature_cols].fillna(0)

            # Split
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

            # Train
            if is_classification or task_type == "classification":
                model = RandomForestClassifier(n_estimators=50, random_state=42)
                metric = "Accuracy"
            else:
                model = RandomForestRegressor(n_estimators=50, random_state=42)
                metric = "R²"

            model.fit(X_train, y_train)

            # Scores
            train_score = model.score(X_train, y_train)
            test_score = model.score(X_test, y_test)

            # Feature importance
            importance = pd.DataFrame({
                'feature': feature_cols,
                'importance': model.feature_importances_
            }).sort_values('importance', ascending=False)

            # Plot importance
            plt.figure(figsize=(10, 6))
            top_features = importance.head(8)
            plt.barh(range(len(top_features)), top_features['importance'].values)
            plt.yticks(range(len(top_features)), top_features['feature'].values)
            plt.xlabel('Importance')
            plt.title('Top 8 Feature Importance')
            plt.tight_layout()

            # Save plot
            img_bytes = io.BytesIO()
            plt.savefig(img_bytes, format='png', dpi=100)
            img_bytes.seek(0)
            plt.close()

            img_base64 = base64.b64encode(img_bytes.read()).decode()
            html_img = f'<img src="data:image/png;base64,{img_base64}" style="width:100%">'

            result = f"""
### 🎯 ML Results
**Target:** {target_col}
**Model:** Random Forest
**Samples:** {len(X):,}
**Features:** {len(feature_cols)}

**Performance:**
- Training {metric}: {train_score:.3f}
- Test {metric}: {test_score:.3f}

**Top 3 Features:**
1. {importance.iloc[0]['feature']}: {importance.iloc[0]['importance']:.3f}
2. {importance.iloc[1]['feature']}: {importance.iloc[1]['importance']:.3f}
3. {importance.iloc[2]['feature']}: {importance.iloc[2]['importance']:.3f}
            """

            return result, html_img, importance

        except Exception as e:
            return f"Error: {str(e)}", None, pd.DataFrame()

# ============================================
# CREATE GRADIO APP
# ============================================

# Initialize
app = SimpleSaaS()

# Create interface
with gr.Blocks(title="AI SaaS Platform", theme='default') as demo:

    gr.Markdown("""
    # 🚀 AI SaaS Platform
    ### Upload CSV → Instant Insights + Predictions
    """)

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="📁 Upload CSV", file_types=[".csv"])
            load_btn = gr.Button("📤 Load Data", variant="primary")

        with gr.Column(scale=2):
            status = gr.Textbox(label="Status", lines=2)

    with gr.Row():
        rows_box = gr.Number(label="Rows", value=0)
        cols_box = gr.Number(label="Columns", value=0)
        target_dropdown = gr.Dropdown(label="Target Column", choices=[], interactive=True)

    preview = gr.Dataframe(label="Data Preview", height=200)

    with gr.Tabs():
        # Tab 1: Analysis
        with gr.TabItem("📊 Analysis"):
            analyze_btn = gr.Button("🔍 Run Analysis", variant="primary")

            with gr.Row():
                with gr.Column(scale=1):
                    analysis_text = gr.Markdown()
                with gr.Column(scale=1):
                    analysis_plot = gr.HTML()

        # Tab 2: ML
        with gr.TabItem("🤖 ML"):
            with gr.Row():
                task = gr.Radio(["auto", "classification", "regression"],
                               value="auto", label="Task Type")
                ml_btn = gr.Button("🚀 Train Model", variant="primary")

            with gr.Row():
                with gr.Column(scale=1):
                    ml_text = gr.Markdown()
                with gr.Column(scale=1):
                    ml_plot = gr.HTML()

            ml_table = gr.Dataframe(label="Feature Importance", height=200)

        # Tab 3: Chat
        with gr.TabItem("💬 Chat"):
            chat_input = gr.Textbox(label="Ask something",
                                   placeholder="e.g., How many rows?")
            chat_btn = gr.Button("Ask", variant="primary")
            chat_output = gr.Textbox(label="Response", lines=5)

    # ============================================
    # EVENT HANDLERS
    # ============================================

    # Load data
    def load_data(file):
        msg, preview, rows, cols, cols_list = app.load_file(file)
        return msg, preview, rows, cols, gr.Dropdown(choices=cols_list)

    load_btn.click(
        load_data,
        inputs=[file_input],
        outputs=[status, preview, rows_box, cols_box, target_dropdown]
    )

    # Analyze
    analyze_btn.click(
        app.analyze_data,
        inputs=[],
        outputs=[analysis_text, analysis_plot]
    )

    # ML
    ml_btn.click(
        app.run_ml,
        inputs=[target_dropdown, task],
        outputs=[ml_text, ml_plot, ml_table]
    )

    # Chat
    def chat_response(question):
        if app.df is None:
            return "Please upload data first!"

        q = question.lower()
        if 'row' in q or 'shape' in q:
            return f"Dataset has {app.df.shape[0]:,} rows and {app.df.shape[1]:,} columns"
        elif 'column' in q:
            cols = list(app.df.columns[:10])
            return f"Columns: {', '.join(cols)}" + (f"... and {len(app.df.columns)-10} more" if len(app.df.columns) > 10 else "")
        elif 'missing' in q:
            return f"Missing values: {app.df.isnull().sum().sum():,}"
        else:
            return "Try asking: rows, columns, missing values"

    chat_btn.click(
        chat_response,
        inputs=[chat_input],
        outputs=[chat_output]
    )

    chat_input.submit(
        chat_response,
        inputs=[chat_input],
        outputs=[chat_output]
    )

# ============================================
# LAUNCH
# ============================================

print("\n" + "="*50)
print("🚀 Starting AI SaaS Platform...")
print("📱 Public URL will appear below")
print("="*50 + "\n")

demo.launch(share=True, debug=False)

✅ Gradio version: 3.40.0
✅ Pandas version: 2.3.3

🚀 Starting AI SaaS Platform...
📱 Public URL will appear below



/tmp/ipykernel_15226/338150251.py:218: GradioDeprecationWarning: `height` is deprecated in `Interface()`, please use it within `launch()` instead.
  preview = gr.Dataframe(label="Data Preview", height=200)
/tmp/ipykernel_15226/338150251.py:244: GradioDeprecationWarning: `height` is deprecated in `Interface()`, please use it within `launch()` instead.
  ml_table = gr.Dataframe(label="Feature Importance", height=200)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
IMPORTANT: You are using gradio version 3.40.0, however version 4.44.1 is available, please upgrade.
--------
Running on public URL: https://2045ee968179c89c21.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [6]:

import pandas as pd
import numpy as np

# Sample dataset
df = pd.DataFrame({
    'age': np.random.randint(22, 65, 500),
    'salary': np.random.normal(60000, 20000, 500).round(2),
    'experience': np.random.randint(0, 35, 500),
    'department': np.random.choice(['Sales', 'Marketing', 'Engineering', 'HR', 'Finance'], 500),
    'performance_score': np.random.uniform(1, 10, 500).round(2),
    'satisfaction': np.random.uniform(1, 5, 500).round(2),
    'left_company': np.random.choice([0, 1], 500, p=[0.8, 0.2])  # Target for classification
})

df.to_csv('employee_data.csv', index=False)
print("✅ employee_data.csv created!")

# Download karne ke liye
from google.colab import files
files.download('employee_data.csv')

✅ employee_data.csv created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# Sales prediction
df = pd.DataFrame({
    'product_id': [f'P{str(i).zfill(3)}' for i in range(1, 501)],
    'price': np.random.uniform(10, 1000, 500).round(2),
    'discount': np.random.uniform(0, 30, 500).round(1),
    'marketing_spend': np.random.uniform(100, 5000, 500).round(2),
    'store_rating': np.random.uniform(3, 5, 500).round(1),
    'holiday_season': np.random.choice([0, 1], 500),
    'units_sold': np.random.randint(10, 500, 500)  # Target for regression
})
df.to_csv('sales_data.csv', index=False)
files.download('sales_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:

import pandas as pd
df = pd.read_csv('employee_data.csv')
print("✅ File valid!")
print(df.info())
print(df.head())

✅ File valid!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                500 non-null    int64  
 1   salary             500 non-null    float64
 2   experience         500 non-null    int64  
 3   department         500 non-null    object 
 4   performance_score  500 non-null    float64
 5   satisfaction       500 non-null    float64
 6   left_company       500 non-null    int64  
dtypes: float64(3), int64(3), object(1)
memory usage: 27.5+ KB
None
   age    salary  experience   department  performance_score  satisfaction  \
0   30  62941.06          11      Finance               7.95          2.27   
1   59  59970.24           1           HR               5.95          1.77   
2   34  71419.07          19  Engineering               2.58          2.83   
3   30  71164.67          23        Sales               1.19          